In [6]:
"""
Colorado Reservoirs — Storage Modeling Pipeline (generalized from Blue Mesa)
=============================================================================
Same STAGE 1 pipeline as Blue_Mesa_Resevoir_Corresponding_Months_Data.ipynb,
extended to 3 of the 10 reservoirs in the watershed/SNOTEL comparison. Logic
is unchanged -- pull daily storage, aggregate to per-water-year June/July
features plus the Oct-1 antecedent predictor -- just looped over reservoirs
instead of hardcoded to Blue Mesa's RISE item IDs.

STAGE 1 (this script): for each reservoir, pull daily storage, aggregate to
         per-water-year June mean / July mean / June-July mean, plus the
         Oct-1 antecedent-storage predictor. Saves one reservoir_features.csv
         per reservoir + one combined table, and triggers a browser download
         of the combined table.

STAGE 2 (team): merge with teammate predictors (SWE, temp, NDVI, precip),
         split train/test, fit OLS (+ Ridge/Lasso), report R2/RMSE/VIF.

Run in Colab. Needs network access to data.usbr.gov (RISE).

Water-year convention (unchanged from the original):
    WY{y} = Oct 1 {y-1} 00:00  ->  Sep 30 {y}.
    June/July of WY{y} are calendar June/July {y}; the Oct-1 antecedent for
    WY{y} is storage around Oct 1 {y-1} (carryover BEFORE this year's snow).

EXCLUDED -- McPhee, Dillon, John Martin, Vallecito, Lake Granby,
Green Mountain, Twin Lakes:
  - McPhee is in RISE, but its storage item ID couldn't be found despite
    extensive searching (data.usbr.gov's catalog isn't fully crawlable).
  - Dillon isn't a Reclamation facility (Denver Water owns it), so it's not
    in RISE; USGS NWIS doesn't publish storage for its site either; and
    Colorado's own DWR/CDSS system rejected every plausible query parameter
    tried against it.
  - John Martin (Army Corps, routed through USGS NWIS) is excluded on data
    quality: its storage record stops in Oct 2020, leaving 5 of 41 water
    years (2021-2025) with zero June/July data.
  - Vallecito is dropped on SWE-record depth, not storage-data quality: no
    SNOTEL station in its contributing watershed (Pine/Los Pinos River)
    reports before 1985-10-01 (the Vallecito site itself). Nearby sites
    that report earlier -- Columbus Basin (1994), Stump Lakes (1985) -- are
    later still or sit in the adjacent Florida River drainage, not Pine
    River. That leaves no SWE predictor with coverage back to WY1980,
    unlike the reservoirs kept below, which all have an in-basin SNOTEL
    site reporting by 1978-1979.
  - Lake Granby, Green Mountain, and Twin Lakes were provisionally kept
    over the shorter 1985-2025 window (missing days under 5%: Granby
    0.47%, Green Mountain 0.81%) but fail the same missing-data standard
    once the window was extended back to 1980 -- confirmed against the
    actual STAGE 1 run:
      * Lake Granby: 1,894 of 16,772 calendar days missing (~11.3%), plus
        thin (<20 days) June/July coverage in WY1981-1985.
      * Green Mountain: 1,944 days missing (~11.6%, after dropping 14
        duplicate-timestamp rows on the elevation item), plus thin
        June/July coverage in WY1981-1985 and WY1998.
      * Twin Lakes: the aligned daily record doesn't even start until
        1980-12-16 (misses the WY1980 antecedent Oct-1 1979 window
        entirely), 1,096 days missing (~7.2%) after dropping 90
        duplicate-timestamp rows, and thin June/July coverage in WY1980,
        1984-1986.
    Applying the >5%-missing / no-thin-years standard consistently across
    the full 1980-2025 window, not just over whatever window a reservoir
    happened to look clean in.
  All 3 remaining reservoirs (Blue Mesa, Navajo, Pueblo) pull from RISE
  only, with clean daily coverage across the full 1980-2025 window.
"""

from google.colab import drive
import os

# Same verified-mount pattern as the watershed-extraction script: a plain
# mkdir succeeds even when Drive isn't actually mounted, silently writing to
# ephemeral VM storage instead. This checks the mount is real before
# writing anything.
if not os.path.ismount("/content/drive"):
    print("Mounting Google Drive (approve the popup)...")
    drive.mount("/content/drive")
if not os.path.ismount("/content/drive"):
    raise RuntimeError(
        "Google Drive did not mount. Re-run this cell and approve the auth "
        "popup -- refusing to fall back to local VM storage, which "
        "disappears on disconnect."
    )

OUT_DIR_ROOT = "/content/drive/MyDrive/esiil-2026/reservoirs"
os.makedirs(OUT_DIR_ROOT, exist_ok=True)
print(f"Output root: {OUT_DIR_ROOT}")

Output root: /content/drive/MyDrive/esiil-2026/reservoirs


In [7]:
# -----------------------------------------------------------------------------
# 0. SETUP
# -----------------------------------------------------------------------------
!pip install pandas numpy requests scikit-learn statsmodels --quiet

import io
import numpy as np
import pandas as pd
import requests

try:
    from google.colab import files as _gfiles
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RISE_BASE = "https://data.usbr.gov/rise/api/result/download"

START_YEAR = 1980          # first water year (antecedent Oct-1 predictor
                           # for WY1980 pulls from Oct 1, 1979)
END_YEAR   = 2025          # last water year
TRAIN_END  = 2009          # train = WY 1980..2009  (see boundary note below)
TEST_START = 2010          # test  = WY 2010..2025

# NOTE ON THE 1980-2010 / 2010-2025 SPLIT THE TEAM WROTE:
# WY 2010 appears in BOTH ranges as written. That's leakage. This script puts
# 2010 in TEST only (train <= 2009). Raise this with Mia/Armando so the whole
# team uses one convention. Change TRAIN_END/TEST_START here if they disagree.

# -----------------------------------------------------------------------------
# Reservoir registry -- all 3 pull from RISE. McPhee, Dillon, John Martin,
# Vallecito, Lake Granby, Green Mountain, Twin Lakes excluded -- see module
# docstring (the last three failed the >5%-missing / no-thin-years standard
# once the window was extended back to 1980).
# -----------------------------------------------------------------------------
RESERVOIRS = {
    "blue_mesa": dict(
        name="Blue Mesa Reservoir", source="rise",
        storage_item=76, area_item=4773, elev_item=78,
    ),
    "navajo": dict(
        name="Navajo Reservoir", source="rise",
        storage_item=613, area_item=4785, elev_item=612,
    ),
    "pueblo": dict(
        name="Pueblo Reservoir", source="rise",
        storage_item=681, area_item=None, elev_item=682,
    ),
}

In [8]:
# -----------------------------------------------------------------------------
# 1. PULL DAILY STORAGE -- RISE or USGS depending on the reservoir
# -----------------------------------------------------------------------------
def rise_csv(item_id):
    """
    Full daily series for a RISE catalog item via the CSV download endpoint.
    Returns a pandas Series indexed by date. Unchanged from the Blue Mesa
    notebook -- the CSV endpoint is the proven pull used there.
    """
    url = f"{RISE_BASE}?type=csv&itemId={item_id}"
    lines = requests.get(url, timeout=300).text.splitlines()

    try:
        h = next(i for i, l in enumerate(lines)
                 if 'Result' in l and not l.lstrip('"').startswith('#'))
    except StopIteration:
        print(f"item {item_id} — couldn't find data header. First 40 lines:")
        print("\n".join(lines[:40]))
        raise

    df = pd.read_csv(io.StringIO("\n".join(lines[h:])))
    df.columns = [c.strip() for c in df.columns]
    dcol = next(c for c in df.columns
                if any(k in c.lower() for k in ('datetime', 'date', 'timestamp')))
    rcol = next(c for c in df.columns if c.lower() == 'result')

    idx = pd.to_datetime(df[dcol], errors='coerce').dt.normalize()
    s = pd.Series(pd.to_numeric(df[rcol], errors='coerce').values, index=idx).dropna()
    s = s[~s.index.isna()].sort_index()
    # Some RISE items (observed on Green Mountain & Twin Lakes) carry
    # duplicate timestamps -- e.g. a revised/corrected reading on the same
    # calendar date. pd.concat(axis=1) with a duplicate-labeled index raises
    # "cannot reindex on an axis with duplicate labels". Keep the LAST value
    # per date (most likely the corrected reading, since the source is
    # already date-sorted).
    if s.index.duplicated().any():
        n_dupes = s.index.duplicated().sum()
        s = s[~s.index.duplicated(keep='last')]
        print(f"  item {item_id}: dropped {n_dupes} duplicate-date row(s) (kept last)")
    return s


def pull_reservoir_gauge(slug, cfg, start_year, end_year):
    """
    Pull elevation/storage/area daily series and align into one daily
    DataFrame `g` with columns ['elev_ft', 'stor_af', 'area_acres'], sliced
    to the study window. Same structure as the original notebook's `g`.
    Missing series (e.g. no area item available) come back as all-NaN
    columns -- harmless, since only stor_af feeds the actual feature
    pipeline. All 3 remaining reservoirs are RISE-sourced.
    """
    if cfg["storage_item"] is None:
        raise RuntimeError(
            f"{cfg['name']}: no RISE storage_item on file (see module "
            f"docstring) -- look it up at data.usbr.gov/time-series/search "
            f"and fill it in, then re-run."
        )
    empty = pd.Series(dtype=float, index=pd.DatetimeIndex([]))
    stor = rise_csv(cfg["storage_item"])
    elev = rise_csv(cfg["elev_item"]) if cfg.get("elev_item") else empty
    area = rise_csv(cfg["area_item"]) if cfg.get("area_item") else empty

    g = pd.concat([elev, stor, area], axis=1)
    g.columns = ['elev_ft', 'stor_af', 'area_acres']
    g = g.loc[f"{start_year-1}-09-01":f"{end_year}-08-01"]
    if g['stor_af'].dropna().empty:
        raise RuntimeError(f"{cfg['name']}: no storage values in range — check source/IDs.")
    print(f"  pulled {len(g):,} aligned daily rows "
          f"{g.index.min().date()} – {g.index.max().date()}")
    return g


def daily_storage_frame(g):
    """Adapt `g` to the ['date','value'] frame the aggregator expects (value=stor_af)."""
    s = g['stor_af'].dropna()
    return (pd.DataFrame({"date": s.index, "value": s.values})
              .drop_duplicates("date").sort_values("date").reset_index(drop=True))


def qa_daily(df):
    """Quick QA pass: report gaps and suspicious flatlines. Returns df unchanged."""
    full = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    missing = len(full) - df["date"].nunique()
    run = (df["value"].diff() != 0).cumsum()
    longest_flat = df.groupby(run).size().max()
    print(f"  QA: {len(df):,} daily records | {missing:,} calendar days missing "
          f"| longest identical-value run = {int(longest_flat)} days")
    if missing > 0.05 * len(full):
        print("  QA WARNING: >5% of days missing — inspect before trusting means.")
    return df

In [9]:
# -----------------------------------------------------------------------------
# 2. AGGREGATE TO PER-WATER-YEAR FEATURES (unchanged from the original)
# -----------------------------------------------------------------------------
def build_reservoir_features(daily, start_year, end_year):
    """
    For each water year, compute:
      jun_mean_af  : mean daily storage over June {y}
      jul_mean_af  : mean daily storage over July {y}
      jun_jul_af   : mean daily storage over June 1 - July 31 {y}  (single window)
      antecedent_oct1_af : 7-day centered mean storage around Oct 1 {y-1}
      n_days_jun, n_days_jul : record counts (QA — flag thin coverage)
    """
    daily = daily.set_index("date")
    recs = []
    for wy in range(start_year, end_year + 1):
        jun = daily.loc[f"{wy}-06-01":f"{wy}-06-30", "value"]
        jul = daily.loc[f"{wy}-07-01":f"{wy}-07-31", "value"]
        jj  = daily.loc[f"{wy}-06-01":f"{wy}-07-31", "value"]

        oct1 = pd.Timestamp(f"{wy-1}-10-01")
        ant = daily.loc[oct1 - pd.Timedelta(days=3): oct1 + pd.Timedelta(days=3), "value"]

        recs.append({
            "water_year": wy,
            "jun_mean_af": jun.mean(),
            "jul_mean_af": jul.mean(),
            "jun_jul_af": jj.mean(),
            "antecedent_oct1_af": ant.mean() if len(ant) else np.nan,
            "n_days_jun": int(jun.notna().sum()),
            "n_days_jul": int(jul.notna().sum()),
        })
    out = pd.DataFrame(recs)
    thin = out[(out.n_days_jun < 20) | (out.n_days_jul < 20)]
    if len(thin):
        print(f"  NOTE: {len(thin)} water-year(s) have <20 days in Jun or Jul "
              f"(thin coverage): {list(thin.water_year)}")
    return out


def run_stage1_for(slug, cfg):
    """Run STAGE 1 for one reservoir, save its outputs, return the features df."""
    print(f"\nSTAGE 1 — {cfg['name']} ({cfg['source']})...")
    out_dir = f"{OUT_DIR_ROOT}/{slug}"
    os.makedirs(out_dir, exist_ok=True)

    g = pull_reservoir_gauge(slug, cfg, START_YEAR, END_YEAR)
    g.to_csv(f"{out_dir}/gauge_daily.csv")
    daily = daily_storage_frame(g)
    qa_daily(daily)
    feats = build_reservoir_features(daily, START_YEAR, END_YEAR)
    feats.insert(0, "reservoir", cfg["name"])
    feats.to_csv(f"{out_dir}/reservoir_features.csv", index=False)
    print(f"  Saved to {out_dir}/")
    return feats


def run_stage1_all():
    all_feats = []
    skipped = []
    for slug, cfg in RESERVOIRS.items():
        try:
            all_feats.append(run_stage1_for(slug, cfg))
        except Exception as e:
            print(f"  SKIPPED {cfg['name']}: {e}")
            skipped.append(cfg["name"])

    if not all_feats:
        raise RuntimeError("No reservoirs succeeded -- check network/IDs before going further.")

    combined = pd.concat(all_feats, ignore_index=True)
    combined_path = f"{OUT_DIR_ROOT}/reservoir_features_all.csv"
    combined.to_csv(combined_path, index=False)

    print(f"\n{'='*60}")
    print(f"Done: {len(all_feats)}/{len(RESERVOIRS)} reservoirs pulled successfully.")
    if skipped:
        print(f"Skipped ({len(skipped)}): {', '.join(skipped)}")
    print(f"Combined table: {combined_path}")
    print(combined.groupby("reservoir")["jun_jul_af"].agg(["count", "mean"]).to_string())

    if IN_COLAB:
        print("\nTriggering browser download of reservoir_features_all.csv ...")
        _gfiles.download(combined_path)
    else:
        print("\nNot in Colab -- skipping browser download, file is already on local disk.")

    return combined

In [10]:
# -----------------------------------------------------------------------------
# 3. RUN IT
# -----------------------------------------------------------------------------
all_feats = run_stage1_all()


STAGE 1 — Blue Mesa Reservoir (rise)...
  pulled 16,772 aligned daily rows 1979-09-01 – 2025-08-01
  QA: 16,772 daily records | 0 calendar days missing | longest identical-value run = 4 days
  Saved to /content/drive/MyDrive/esiil-2026/reservoirs/blue_mesa/

STAGE 1 — Navajo Reservoir (rise)...
  pulled 16,772 aligned daily rows 1979-09-01 – 2025-08-01
  QA: 16,772 daily records | 0 calendar days missing | longest identical-value run = 9 days
  Saved to /content/drive/MyDrive/esiil-2026/reservoirs/navajo/

STAGE 1 — Pueblo Reservoir (rise)...
  pulled 16,768 aligned daily rows 1979-09-01 – 2025-08-01
  QA: 16,768 daily records | 4 calendar days missing | longest identical-value run = 5 days
  Saved to /content/drive/MyDrive/esiil-2026/reservoirs/pueblo/

Done: 3/3 reservoirs pulled successfully.
Combined table: /content/drive/MyDrive/esiil-2026/reservoirs/reservoir_features_all.csv
                     count          mean
reservoir                               
Blue Mesa Reservoir   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>